# Definition of Site Parameters

## About European Spallation Source
The European Spallation Source (ESS) is a European Research Infrastructure Consortium (ERIC), a multi-disciplinary research facility based on the world’s most powerful neutron source. Thier global vision is to build and operate the world’s most powerful accelerator-based neutron source, enabling scientific breakthroughs in research related to 
 * materials
 * energy
 * health 
 * environment
 Additionally, the research carried out at the ESS is envisonded to address some of the most important societal challenges of our time.

### ESS Logo
<img src="https://ess.eu/themes/custom/ess/logo.svg" alt="ESS logo" width="300">

### ESS Location
The European Spallation Source is under construction on the outskirts of Lund, a city in southern Sweden.
The geographic paramters of the site are listed below:

* Longitude: 13.248
* Latitude: 55.734
* Altitude: 76


<img src="https://ess.eu/sites/default/files/styles/article_image/public/images/media/2024-03/DJI_0269.jpg?itok=ZSlvwWan" alt="ESS Ariel View" width="800">


In [ ]:
import pvlib
from pvlib.pvsystem import PVSystem
from pvlib.location import Location
from pvlib.modelchain import ModelChain

from tools.weather import *
from tools.solar_plot import *
from tools.models import FinancialModels

import plotly.express as px


from IPython.display import display


%load_ext autoreload
%autoreload 2

In [ ]:
##########SITE PARAMETERS############
name = "Lund"
site = "ESS-Lund"
altitude = 76 # in meters
longitude = 13.248
latitude = 55.734
tz='Europe/Paris'
building_height = 9
available_surface = 70000  # value in m²
altitude += building_height
##########PV INSTALLATION PARAMETERS############

surface_tilt = 41
surface_azimuth = 180 # pvlib uses 0=North, 90=East, 180=South, 270=West convention
albedo = pvlib.albedo.SURFACE_ALBEDOS["concrete"]

data_source = "openmeteo"       # options are "openmeteo" or "PVGIS"

renderer = "notebook"  # options are "notebook" or "browser"

In [ ]:
# Create a pvlib location object
location = Location(name = name , latitude=latitude, longitude=longitude, altitude=altitude, tz=tz,)
location

## Add the modules
in the next cell , the PV module and Inverter module are selected as :

* Inverter : [SMA_America__SB10000TL_US__240V_](https://www.solaris-shop.com/sma-sunny-boy-sb10000tl-us-12-10kw-string-inverter/?srsltid=AfmBOooQM-9rbcLe9KFdQQxR1_xMD6_1ds2GNlUjxUX9YP4dD2ZhfGSh)
* PV Module: [Jinko_Solar_Co___Ltd_JKM410M_72HL](https://www.solarelectricsupply.com/jinko-eagle-g2-jkm410m-72hl-v-410w-perc-solar-panel?srsltid=AfmBOoq6GapbtrAL_kjIl9OYEN_FFcP1gbMhmMbW8sL93WkZqHQq3yzn)

**Note that these modules are not the latest and might be outdated, however the resulting production curves are still relevant**

In [ ]:
############ PV MODULE ############

pv_modules = pvlib.pvsystem.retrieve_sam('CECMod')
cec_module = pv_modules["Jinko_Solar_Co___Ltd_JKM410M_72HL"]
panel_area = cec_module.A_c

############ INVERTER ############
cec_inverters = pvlib.pvsystem.retrieve_sam('cecinverter')
cec_inverter = cec_inverters['SMA_America__SC_2750_EV_US__600V_']

############ SYSTEM DIMENSIONING ############
mod_per_string = round(cec_inverter.Vdcmax/cec_module.V_oc_ref, 0) -1
total_panels = round(cec_inverter.Pdco/cec_module.PTC, 0)
strings = total_panels/mod_per_string



total_surface_per_inv = total_panels * panel_area
no_inverters = int(available_surface / total_surface_per_inv)


print(f"\nTotal number of PV modules : {total_panels}")
print(f"\nTotal Number of PV panels per string : {mod_per_string}")
print(f"\nTotal number of strings : {strings} ")
print (f"\nInput Power (DC) STC per inverter: {round((total_panels *cec_module.STC)/1000,2) }kW")
print (f"\nInput Power (DC) PTC per inverter: {round((total_panels *cec_module.PTC)/1000, 2 )}kW")
print (f"\nTotal number of invertes: {no_inverters}")


### 3D Representation of the Site 

In [ ]:
plot = False

if plot:
    radius = 500  # meters
    zoom = 15
    import math
    
    new_building = {
    'coordinates': [
        [
            (13.244860, 55.733250),
            (13.244772, 55.733388),
            (13.245062, 55.733447),
            (13.244930, 55.733605),
            (13.245349, 55.733732),
            (13.245453, 55.733643),
            (13.245418, 55.733852),
            (13.245945, 55.734014),
            (13.246129, 55.733825),
            (13.245748, 55.733692),
            (13.245830, 55.733620),
            (13.245941, 55.733712),
            (13.246181, 55.733523),
            (13.245822, 55.733367),
            (13.245741, 55.733420),
            (13.245598, 55.733355),
            (13.245591, 55.733373),
            (13.245651, 55.733316),
            (13.245329, 55.733186),
            (13.245184, 55.733293),
            (13.244860, 55.733250)
        ]
    ],
    'elevation': 3,
    'name': 'Manual Building'
    }


    plotter = SolarSite3DPlot(
        latitude=latitude,
        longitude=longitude,
        mapbox_api_key=None,
        radius=radius,
        zoom=zoom,
        retries=5,
        backoff_factor=1.5,
        request_timeout=200,
        overpass_query_timeout=180,
        overpass_url="https://overpass-api.de/api/interpreter",
    )

    solar = [f"Building_{i}" for i in [13,14,15,16,18,19,21,22,25,26,27,36,37,38,45,50,58,60,64]]
    solar.extend(["Manual Building", "Ship 1", "Ship 2", "Ship 3","Space","Science Center(2026)", "The Loop"])
    deck = plotter.show(
        solar_roof_names=solar,
        manual_building=new_building,
    )

    deck.show()

## Request for Weather Data

The weather data is sourced from [open-meteo](https://open-meteo.com/) or [PVGIS](https://joint-research-centre.ec.europa.eu/photovoltaic-geographical-information-system-pvgis_en)

The following paramters are extracted from ``Open-Meteo``:

* Relative humidity [%]
* Pressure [hpa]
* Cloud cover [%]
* Wind direction [°]
* Ambient Temperature [°C]
* Wind speed [m/s]
* GHI [W/m²]
* DNI [W/m²]
* DHI [W/m²]
* Direct radiation [W/m²]
* Rain [mm]
* Snowfall [cm]

The following paramters are extracted from ``PV GIS``
* Wind speed [m/s]
* GHI [W/m²]
* DNI [W/m²]
* DHI [W/m²]
* Ambient Temperature [°C]
* Wind speed [m/s]
* Wind direction [°]
* Pressure [hpa]

In [ ]:
year = 2025
start_date = f"2020-01-01"
end_date =  f"{year}-12-31"
w = WeatherData(latitude = latitude, longitude=longitude, start_date = start_date, end_date = end_date, timezone="Europe/Stockholm", site = site)
if data_source == "openmeteo":


  
    variables = ("temperature_2m","relative_humidity_2m","rain","snowfall","surface_pressure",
                "cloud_cover","wind_speed_10m","wind_direction_10m","shortwave_radiation",
                "direct_radiation","diffuse_radiation","direct_normal_irradiance" )

    weather = w.get_open_meteo_data(wind_speed_unit= "ms",variables = variables, retries=5, verbose=False )

elif data_source == "PVGIS":
    weather = w.get_pvgis_data(coerce_year= year)
# fig = px.line (weather.loc[weather.index.year == year], template = "plotly_white", title = f"Weather data for {year} [Data Source: {data_source}]")

# fig.show(renderer = renderer)

### Create a Temperature model for the PV Modules 

In [ ]:

temperature_model_parameters = pvlib.temperature.TEMPERATURE_MODEL_PARAMETERS['sapm']['open_rack_glass_polymer']

temperature_model_parameters

In [ ]:
system = PVSystem(
                 surface_tilt=surface_tilt, 
                 surface_azimuth=surface_azimuth,
                 module_parameters=cec_module,
                 inverter_parameters=cec_inverter,
                 temperature_model_parameters=temperature_model_parameters,
                 albedo=albedo, 
                 surface_type=None, 
                 module=None, 
                 module_type='glass_polymer',
                 modules_per_string=mod_per_string, 
                 strings_per_inverter=strings, 
                 inverter=None, 
                 racking_model='open_rack', 
                 losses_parameters=None, 
                 name='ESS_Lund'
                 )

In [ ]:
mc = ModelChain(
                system, location, name=name, 
                clearsky_model='ineichen',
                transposition_model='haydavies',
                solar_position_method='nrel_numpy', 
                airmass_model='kastenyoung1989',
                dc_model="cec", #from module specifications
                ac_model='sandia', #from inverter specifications
                aoi_model='physical', 
                spectral_model='no_loss', #spectral model 'no loss' otherwise error
                temperature_model='sapm',
                 losses_model='pvwatts'
                 )

## Run the Model with clear sky irradiation data
Here, the model does not consider cloud cover and assumes maximum irradiation all the time

In [ ]:
mc.run_model(weather)

In [ ]:
dc = mc.results.dc.p_mp.tolist()
ac = mc.results.ac.tolist()
c_t = mc.results.cell_temperature.tolist()


# ind = 
df = pd.DataFrame({"Power_dc" : dc , "Power_ac" : ac , "cell_temp" : c_t}, index= mc.results.times)

df.index = df.index.tz_localize(None)
df = df.loc[start_date: ]
df.loc[:, ["Power_dc", "Power_ac"]] *= no_inverters

df_plot = df.loc[df.index>="2025"]


fig = px.line(df_plot[["Power_dc", "Power_ac" ]]/1000, template = "plotly_white", title = "Hourly PV Production [W]")
fig.update_yaxes(title_text="Power [kW]")
fig.show(renderer = "notebook")
# fig.show(renderer = "png")

fig_1 = px.line(df_plot[["cell_temp" ]], template = "plotly_white", title = "Hourly PV Cell Temperature [°C]")
fig_1.update_yaxes(title_text="Temperature [°C]")
fig_1.show(renderer = "notebook")
# fig_1.show(renderer = "png")

fig_2 = px.bar(df[["Power_dc", "Power_ac" ]].resample("1YE").sum()/1e6,barmode ="group", template = "plotly_white", title = "Estimated Annual Energy production - Clear Sky [GWh]" )
fig_2.update_yaxes(title_text="Energy [MWh]")
fig_2.show(renderer = "notebook")


# Accounting for cloud cover
code below replicates the forecast method for GHI as depicted in [readthedocs](https://pvlib-python.readthedocs.io/en/v0.5.2/_modules/pvlib/forecast.html#ForecastModel.cloud_cover_to_ghi_linear) for the ```cloud_cover_to_ghi_linear``` method. Subsequently the DNI and DHI are recalculated using the ```erbs``` function from pvlib

In [ ]:
weather_1 = weather.copy()
offset = 0.35 

solar_position = location.get_solarposition(weather_1.index)
zenith = solar_position["apparent_zenith"]



weather_1["ghi"] *=  (offset + (1 - offset) * (1 - weather_1["cloud_cover"] / 100))


# weather_1["ghi"] *=  (1 - weather_1["cloud_cover"] / 100)
data = pvlib.irradiance.erbs(
        ghi=weather_1["ghi"],
        datetime_or_doy = weather_1.index,
        zenith=zenith,
    )

weather_1["dni"] = data.dni
weather_1["dhi"] = data.dhi
mc.run_model(weather_1)

In [ ]:
plot = False
dc = mc.results.dc.p_mp.tolist()
ac = mc.results.ac.tolist()
c_t = mc.results.cell_temperature.tolist()
# ind = 
df_1 = pd.DataFrame({"Power_dc" : dc , "Power_ac" : ac , "cell_temp" : c_t}, index= mc.results.times)

df_1.index = df_1.index.tz_localize(None)
df_1 = df_1.loc[start_date: ]

df_1.loc[:, ["Power_dc", "Power_ac"]] *= no_inverters

df_plot = df_1.loc[df_1.index>="2024"]

if plot:
    fig = px.line(df_plot[["Power_dc", "Power_ac" ]]/1000, template = "plotly_white", title = "Hourly PV Production [KW]")
    fig.update_yaxes(title_text="Power [kW]")
    # fig.show(renderer = "notebook")
    # fig.show(renderer = "png")


    fig_1 = px.line(df_plot[["cell_temp" ]], template = "plotly_white", title = "Hourly PV Cell Temperature [°C]")
    fig_1.update_yaxes(title_text="Temperature [°C]")
    # fig_1.show(renderer = "notebook")
    # fig_1.show(renderer = "png")

    fig_2 = px.bar(df_1[["Power_dc", "Power_ac" ]].resample("1YE",label="left").sum()/1e9,barmode ="group", template = "plotly_white", title = "Estimated Annual Energy production - Cloudy Sky[GWh]" )
    fig_2.update_yaxes(title_text="Energy [GWh]")

    fig_2.show(renderer = "notebook")

In [ ]:
temp =pd.DataFrame(index=df.index)

temp ["Power_dc[clear_sky]"] = df.Power_dc
temp ["Power_dc[cloudy_sky]"] = df_1.Power_dc

fig_1 = px.line(temp.resample("1ME",label="left").sum()/1e6, template="plotly_white", title = "Comparison of clear sky and cloudy sky results [MWh]" )
fig_1.update_yaxes(title_text="Energy [MWh]")
# fig_1.update_traces(type="scattergl")
fig_1.show(renderer = renderer)

In [ ]:
import plotly.graph_objs as go

gen = temp.resample("1h").mean().resample("1YE" , label = "left").sum()/1e9
gen = gen.iloc[1:]

fig = go.Figure()
gen

fig.add_trace(go.Scatter(
    x=gen.index,
    y=gen['Power_dc[clear_sky]'],
    mode='lines',
    name='Clear_sky',
    line=dict(color='orange')
))

# Lower line (cloudy)
fig.add_trace(go.Scatter(
    y=gen['Power_dc[cloudy_sky]'],
    x=gen.index,
    mode='lines',
    name='Cloudy_sky',
    line=dict(color='blue'),
    fill='tonexty',        # fill area between this trace and the previous one
    fillcolor='rgba(0,0,255,0.2)'  # semi-transparent blue
))

fig.update_layout(
    title="Clear Sky vs Cloudy Sky",
    xaxis_title="Time",
    yaxis_title="Value",
    template = "plotly_white",
    yaxis = {"title_text" : "Energy [GWh]"}
)

fig.update_yaxes()
# fig.update_traces(type="scattergl")
fig.show()


# Levelized Cost of Energy (LCOE)

## What is LCOE?

LCOE is the constant price per unit of electricity (usually €/MWh or ¢/kWh) that makes the present value of a generator’s costs equal to the present value of the electricity it produces over its lifetime. It is commonly used to compare the average cost of electricity from different technologies.

### Formulation

**Discounted form:**

```math
LCOE = \frac{\sum_{t=0}^{N} \frac{I_t + O_t + F_t}{(1+r)^t}}{\sum_{t=0}^{N} \frac{E_t}{(1+r)^t}}
```

Where:

* $`I_t`$ = investment (capital) expenditures in year t
* $`O_t`$ = operations & maintenance costs in year t
* $`F_t`$ = fuel costs in year t (0 for wind/solar)
* $`E_t`$ = electricity produced in year t (MWh or kWh)
* $`r`$ = discount rate (often WACC)
* $`N`$ = project lifetime (years)

### Key Drivers

* Capital cost (CAPEX) and financing (discount rate)
* Capacity factor (higher CF → lower LCOE for same CAPEX)
* Fuel costs (important for fossil fuels)
* O\&M costs and major replacements
* Lifetime and degradation
* Subsidies, tax credits, carbon pricing

### Uses

* Comparing technology costs (wind vs solar vs gas)
* Policy evaluation (impact of subsidies, carbon pricing)
* Project finance / tariff setting / auctions

### Limitations

* Ignores **time-value of energy** (when electricity is produced)
* Does not include **system integration costs** (grid reinforcement, storage)
* Omits **risk & flexibility contributions**
* Highly sensitive to assumptions (discount rate, capacity factor, lifetime)


In [ ]:



project_lifetime = df_1.resample("1YE").sum().shape[0]       # years
discount_rate = 0.07        # 7% discount rate
degradation_rate = 0.005    # 0.5%/year
capex = 6000000           # initial investment ($)
om = 0.03 * capex                 # annual O&M ($)
periodic_cost = 0.1*capex
periodic_cost_freq = 5

if "temp" in globals() and {"Power_dc[clear_sky]", "Power_dc[cloudy_sky]"}.issubset(temp.columns):
    generation_source = temp
    power_columns = {
        "clear_sky": "Power_dc[clear_sky]",
        "cloudy_sky": "Power_dc[cloudy_sky]",
    }
else:
    generation_source = df_1
    if "Power_dc" in df_1.columns:
        power_columns = {"cloudy_sky": "Power_dc"}
    elif "Power_ac" in df_1.columns:
        power_columns = {"cloudy_sky": "Power_ac"}
    else:
        power_columns = None  # fallback to auto-detection in FinancialModels

fm = FinancialModels(default_discount_rate=float(discount_rate), currency="€")

financial_results = fm.evaluate_from_generation_df(
    generation_df=generation_source,
    power_columns=power_columns,
    power_unit="W",
    energy_freq="1YE",
    energy_label="right",
    project_lifetime=int(project_lifetime),
    capex=float(capex),
    annual_opex=float(globals().get("om", 0.03 * capex)),
    periodic_cost=float(globals().get("periodic_cost", 0.0)),
    periodic_cost_frequency=int(globals().get("periodic_cost_freq", 5)),
    fuel_cost=float(globals().get("fuel_cost", 0.0)),
    discount_rate=float(discount_rate),
    degradation_rate=float(degradation_rate),
    electricity_price=float(globals().get("electricity_price", 0.2)),
    price_escalation_rate=float(globals().get("escalation_rate", 0.02)),
    production_start_year=1,
    baseline_method="mean",
)

financial_kpis = financial_results["kpis"].copy()
financial_cashflow = financial_results["cashflow"].copy()
annual_energy_kwh = financial_results["annual_energy_kwh"].copy()

# -----------------------------
# 4) KPI table
# -----------------------------
kpi_table = financial_kpis.copy()
kpi_table["IRR_pct"] = kpi_table["IRR"] * 100
display(kpi_table.round(4).sort_index())

# -----------------------------
# 5) Plotly: LCOE by scenario
# -----------------------------
fig_lcoe = px.bar(
    financial_kpis.reset_index(),
    x="Scenario",
    y="LCOE_per_kWh",
    template="plotly_white",
    title="LCOE by Scenario",
    labels={"LCOE_per_kWh": "LCOE [€/kWh]"},
)

# fig_lcoe.update_traces(type="scattergl")
fig_lcoe.show(renderer=renderer)



In [ ]:
# Build running LCOE table from existing results
lcoe_evolution = fm.build_lcoe_evolution(
    cashflow=financial_cashflow,
    kpis=financial_kpis,
    discounted=True,
)

# Pick one scenario (cloudy_sky if available)
scenario = "cloudy_sky" if "cloudy_sky" in lcoe_evolution["Scenario"].unique() else lcoe_evolution["Scenario"].iloc[0]
plot_df = lcoe_evolution[lcoe_evolution["Scenario"] == scenario].copy()

final_lcoe = plot_df["Final_LCOE_per_kWh"].iloc[0]

fig_lcoe_evo = px.line(
    plot_df,
    x="Year",
    y="Running_LCOE_per_kWh",
    template="plotly_white",
    title=f"LCOE Evolution - {scenario}",
    labels={"Running_LCOE_per_kWh": "Running LCOE [€/kWh]"},
)
fig_lcoe_evo.add_hline(
    y=final_lcoe,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Final LCOE = {final_lcoe:.3f} €/kWh",
    annotation_position="top right",
)
fig_lcoe_evo.update_traces(mode="lines+markers")
# fig_lcoe_evo.update_traces(type="scattergl")
fig_lcoe_evo.show(renderer=renderer)

# Discounted Payback Period (DPP)

## What is Discounted Payback Period?

The **Discounted Payback Period (DPP)** is the time it takes for the cumulative present value of cash inflows from a project to equal the initial investment, taking into account the **time value of money**. Unlike the regular payback period, DPP considers discounting of future cash flows.

## Formula

```math
DPP = \text{Time when cumulative discounted inflows} = \text{Initial Investment}
```

Step-by-step:

1. Determine the discount rate (r), usually WACC or required rate of return.
2. Calculate the **discounted cash inflow** for each period:

```math
DCF_t = \frac{CF_t}{(1+r)^t}
```

Where:

* $`CF_t`$ = cash inflow at year t
* $`r`$ = discount rate
* $`t`$ = year

3. Sum discounted cash inflows cumulatively until the total equals or exceeds the initial investment.

## Key Points

* Accounts for the **time value of money**, unlike regular payback period.
* Provides a measure of how quickly a project recovers its investment **in present-value terms**.
* Shorter DPP indicates faster recovery and lower risk.

## Limitations

* Does not consider cash flows after the payback period.
* Sensitive to discount rate selection.
* Provides no direct measure of overall profitability (use NPV or IRR for that).



In [ ]:

dcf_cols = [c for c in financial_cashflow.columns if c.startswith("Discounted_Cumulative_Net_CashFlow_")]
if dcf_cols:
    dcf_df = financial_cashflow[dcf_cols].rename(
        columns=lambda c: c.replace("Discounted_Cumulative_Net_CashFlow_", "")
    )
    dcf_long = dcf_df.reset_index().melt(
        id_vars="Year",
        var_name="Scenario",
        value_name="Discounted_Cumulative_Net_CashFlow"
    )

    fig_dcf = px.line(
        dcf_long,
        x="Year",
        y="Discounted_Cumulative_Net_CashFlow",
        color="Scenario",
        template="plotly_white",
        title="Discounted Cumulative Net Cash Flow",
        labels={"Discounted_Cumulative_Net_CashFlow": "Cash Flow [€]"},
    )
    fig_dcf.add_hline(y=0, line_dash="dash", line_color="black")
    fig_dcf.update_traces(mode="lines+markers")
    # fig_dcf.update_traces(type="scattergl")
    fig_dcf.show(renderer=renderer)



# Environmental Indicators

In [ ]:
import os

dirs = ["./data" ]
df_co = pd.DataFrame()

countries = {"FR": "France",
             "NL": "Netherlands",
             "SE": "Sweden",
             "HU": "Hungary",
             "CZ": "Czech Republic"}


df_co = pd.read_csv("./data/carbon_intensity.csv",  header=0,sep = ",", decimal = "." , index_col=0, parse_dates=True)
# df_co.rename(columns=countries, inplace=True)
df_co = df_co.resample("1h").mean()

years = len(df.resample("1YE").sum()) 
rate = 0.05
temp = df_co.copy()
for year in range (1,years,1):
    t = temp.copy()
    t.index = t.index + pd.DateOffset(years=year)
    t *= ((1 - rate) ** year)

    df_co = pd.concat([df_co, t])


df_co.index = pd.date_range(start=start_date, periods = len(df_co), freq = "1h")

df_co = df_co.loc[str(df_1.index[0]): str(df_1.index[-1])]

In [ ]:
carbon_capex = 80000 * (total_panels * no_inverters * (cec_module.PTC/1000))

df_co["Clear_sky_AC"] = (df["Power_dc"]/1000).tolist()
df_co["Cloudy_sky_AC"] = (df_1["Power_dc"]/1000).tolist()

# Compute avoided CO2 for each hour under each scenario
# “SE” and “CZ” are assumed to be emission intensities (kg CO2 per unit power)
df_co["Avoided_Co2_SE_ClearSky"] = df_co["Clear_sky_AC"] * df_co["SE"]
df_co["Avoided_Co2_CZ_ClearSky"]   = df_co["Clear_sky_AC"] * df_co["CZ"]

df_co["Avoided_Co2_SE_CloudySky"] = df_co["Cloudy_sky_AC"] * df_co["SE"]
df_co["Avoided_Co2_CZ_CloudySky"] = df_co["Cloudy_sky_AC"] * df_co["CZ"]

# Initialize columns for cumulative avoided CO2 and payback detection
# df_co = df_co.copy()

scenarios = [
    ("SE_ClearSky", "Avoided_Co2_SE_ClearSky"),
    ("CZ_ClearSky", "Avoided_Co2_CZ_ClearSky"),
    ("SE_CloudySky", "Avoided_Co2_SE_CloudySky"),
    ("CZ_CloudySky", "Avoided_Co2_CZ_CloudySky")
]

for label, avoided_col in scenarios:
    cum_col = f"Cumulative_{label}"
    
    # 1. Compute the cumulative sum of the avoided CO2
    df_co[cum_col] = df_co[avoided_col].cumsum()
    
    # 2. Subtract the carbon capex so that at t=0, it's negative and we see how long until it crosses 0
    df_co[cum_col] = df_co[cum_col] - carbon_capex


In [ ]:
cols = [col for col in df_co.columns if "Cumu" in col]
weather_1 = weather = None

fig = px.line(df_co[cols].resample("1ME").asfreq()/1000,  title = "Carbon Payback")
fig.update_traces(fill='tozeroy')
fig.add_hline(y=0, line_dash='dash', line_color='black')

fig.update_layout(template= 'plotly_white', xaxis = {"title_text" : "Date"}, yaxis = {"title_text" : "CO2 [kGCO2eqv]"})
# fig.update_traces(type="scattergl")
fig.show(renderer=renderer)